# 🎯 官方 metric 本地驗證：baseline vs 改進版的「真分數」

用**官方開源評分套件** [`tracking-cellmot`](https://github.com/royerlab/kaggle-cell-tracking-competition)（CZ Biohub / royerlab）在訓練集上幫兩個版本打**跟排行榜同一把尺**的分數——含我們 proxy 缺的 **node 膨脹懲罰** `J_adj = max(0, J×(1−0.1×node_ratio))`。

> 🌐 需 Internet On（要 `pip install` 官方套件）。本 notebook 不提交。

**重點懸念**：proxy（沒懲罰）說 improved 0.31 ≫ baseline 0.02；但 improved 預測 14088 個 node、真值才 52 個——官方懲罰會不會把它打回去？這格跑完見真章。

In [ ]:
import subprocess, sys
def pip(*a):
    print('pip', *a)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a])
# 官方評分只需 tracksdata + polars（不需 torch）；tracksdata 會帶 zarr>=3、geff、rustworkx
pip('git+https://github.com/royerlab/tracksdata')
pip('--no-deps', 'git+https://github.com/royerlab/kaggle-cell-tracking-competition')
import tracksdata as td, polars as pl
from tracking_cellmot.metrics import evaluate, per_sample_metrics, summarise, node_recall
print('tracksdata', getattr(td, '__version__', '?'), '| metric import OK')

In [ ]:
import os, json
from collections import defaultdict
import numpy as np
import zarr
from scipy.ndimage import uniform_filter, label, distance_transform_edt
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max
from skimage.segmentation import watershed

TRAIN = '/kaggle/input/competitions/biohub-cell-tracking-during-development/train'
SCALE = (1.625, 0.40625, 0.40625)
SCALE_A = np.array(SCALE)
DOWNSAMPLE, PERCENTILE = 4, 90
MAX_LINK_DISTANCE, DIV_DISTANCE, GAP_DISTANCE, WS_MIN_DISTANCE = 15.0, 8.0, 20.0, 2
MAX_SAMPLES = 2

def scaled_pairwise(A, B):
    d = A[:, None, :] - B[None, :, :]
    return np.sqrt(((d * SCALE_A) ** 2).sum(axis=2))

def open_image(p):
    return zarr.open(p, mode='r')['0']

def read_estimated_n_total(geff_path):
    """讀 .geff 的 estimated_number_of_nodes（真值是稀疏的，這是估計的真實總細胞數）。"""
    try:
        a = dict(zarr.open(geff_path, mode='r').attrs)
        def walk(d):
            if isinstance(d, dict):
                for k, v in d.items():
                    if 'estimated_number_of_nodes' in str(k):
                        return v
                    r = walk(v)
                    if r is not None:
                        return r
            return None
        v = walk(a)
        if v is not None:
            return float(v)
    except Exception as e:
        print('n_total read warn:', e)
    return float('nan')

print('readers ready')

In [ ]:
# === 兩種偵測 + pipeline（同前）===
def detect_baseline(vol):
    ds = vol[::DOWNSAMPLE, ::DOWNSAMPLE, ::DOWNSAMPLE]
    sm = uniform_filter(ds.astype(np.float32), size=3)
    binary = sm > np.percentile(sm, PERCENTILE)
    lab, n = label(binary)
    return [np.argwhere(lab == i).mean(0) * DOWNSAMPLE for i in range(1, n + 1) if (lab == i).any()]

def detect_watershed(vol):
    ds = vol[::DOWNSAMPLE, ::DOWNSAMPLE, ::DOWNSAMPLE]
    sm = uniform_filter(ds.astype(np.float32), size=3)
    binary = sm > np.percentile(sm, PERCENTILE)
    if not binary.any():
        return []
    dist = distance_transform_edt(binary)
    peaks = peak_local_max(dist, min_distance=WS_MIN_DISTANCE, labels=binary)
    if len(peaks) == 0:
        return []
    markers = np.zeros(dist.shape, np.int32)
    markers[tuple(peaks.T)] = np.arange(1, len(peaks) + 1)
    lab = watershed(-dist, markers, mask=binary)
    return [np.argwhere(lab == i).mean(0) * DOWNSAMPLE for i in range(1, int(lab.max()) + 1) if (lab == i).any()]

def run_pipeline(arr, n_t, detect_fn, improved):
    nodes, edges = {}, []
    frame_ids, frame_xyz = [], []
    nid = 1
    for t in range(n_t):
        cents = detect_fn(np.asarray(arr[t]))
        ids, xyz = [], []
        for c in cents:
            nodes[nid] = (t, float(c[0]), float(c[1]), float(c[2]))
            ids.append(nid); xyz.append(c); nid += 1
        frame_ids.append(ids); frame_xyz.append(np.array(xyz) if xyz else np.empty((0, 3)))
    has_in, out_count = set(), defaultdict(int)
    for t in range(n_t - 1):
        pid, pc = frame_ids[t], frame_xyz[t]
        cid, cc = frame_ids[t + 1], frame_xyz[t + 1]
        if len(pid) == 0 or len(cid) == 0:
            continue
        D = scaled_pairwise(pc, cc)
        rr, cc2 = linear_sum_assignment(D)
        mp, mc = set(), set()
        for ri, ci in zip(rr, cc2):
            if D[ri, ci] <= MAX_LINK_DISTANCE:
                edges.append((pid[ri], cid[ci])); mp.add(ri); mc.add(ci); has_in.add(cid[ci]); out_count[pid[ri]] += 1
        if improved:
            for ci in range(len(cid)):
                if ci in mc:
                    continue
                dd = np.sqrt((((pc - cc[ci]) * SCALE_A) ** 2).sum(1)); j = int(np.argmin(dd))
                if j in mp and dd[j] <= DIV_DISTANCE and out_count[pid[j]] < 2:
                    edges.append((pid[j], cid[ci])); has_in.add(cid[ci]); out_count[pid[j]] += 1
    if improved:
        has_out = set(out_count.keys())
        for t in range(n_t - 2):
            ends = [(i, n_) for i, n_ in enumerate(frame_ids[t]) if n_ not in has_out]
            starts = [(j, n_) for j, n_ in enumerate(frame_ids[t + 2]) if n_ not in has_in]
            if not ends or not starts:
                continue
            ec = frame_xyz[t][[i for i, _ in ends]]; sc = frame_xyz[t + 2][[j for j, _ in starts]]
            D = scaled_pairwise(ec, sc); rr, cc2 = linear_sum_assignment(D)
            for ri, ci in zip(rr, cc2):
                if D[ri, ci] <= GAP_DISTANCE:
                    edges.append((ends[ri][1], starts[ci][1])); has_out.add(ends[ri][1]); has_in.add(starts[ci][1])
    return nodes, edges

print('pipeline ready')

In [ ]:
# === 把我們的預測轉成 tracksdata 圖（依官方 predict 腳本的寫法）===
def build_graph(nodes, edges):
    items = list(nodes.items())                       # [(nid,(t,z,y,x)),...]
    id2idx = {nid: i for i, (nid, _) in enumerate(items)}
    g = td.graph.InMemoryGraph()
    for key in ['z', 'y', 'x']:
        g.add_node_attr_key(key, pl.Float64, -999999.0)
    tids = g.bulk_add_nodes([{'t': int(t), 'z': float(z), 'y': float(y), 'x': float(x)}
                             for (_, (t, z, y, x)) in items])
    g.add_edge_attr_key('edge_prob', pl.Float64, 0.0)
    ed = [{'source_id': tids[id2idx[s]], 'target_id': tids[id2idx[d]], 'edge_prob': 1.0}
          for s, d in edges if s in id2idx and d in id2idx]
    if ed:
        g.bulk_add_edges(ed)
    return g

print('graph builder ready')

In [ ]:
# === 用官方 metric 對照打分 ===
samples = sorted(d[:-5] for d in os.listdir(TRAIN) if d.endswith('.geff'))[:MAX_SAMPLES]
print('驗證樣本：', samples, '\n')

results = {'baseline': [], 'improved': []}
for name in samples:
    arr = open_image(os.path.join(TRAIN, name + '.zarr'))
    n_t = arr.shape[0]
    gt_res = td.graph.IndexedRXGraph.from_geff(os.path.join(TRAIN, name + '.geff'))
    gt = gt_res[0] if isinstance(gt_res, tuple) else gt_res
    n_total = read_estimated_n_total(os.path.join(TRAIN, name + '.geff'))
    for tag, detect, improved in [('baseline', detect_baseline, False),
                                  ('improved', detect_watershed, True)]:
        try:
            nodes, edges = run_pipeline(arr, n_t, detect, improved)
            g = build_graph(nodes, edges)
            er = evaluate(g, gt, scale=SCALE, max_distance=7.0)
            recall = node_recall(g, gt)
            row = per_sample_metrics(er, n_total, recall)
            results[tag].append(row)
            print(f'{name} [{tag:8}] pred_nodes={len(nodes)} n_total={n_total} | row={row}')
        except Exception as e:
            import traceback; traceback.print_exc()
            print(f'{name} [{tag}] FAILED: {type(e).__name__}: {e}')

print('\n=== 官方 metric：最終分數（含懲罰）===')
for tag in ['baseline', 'improved']:
    if results[tag]:
        s = summarise(results[tag])
        print(f'{tag:8}: score={s.get("score")}  |  {s}')